# Bean Sprout Growth Experiment — Data Analysis
## The Effect of Coloured Light on Mung Bean Sprout Growth
**CID: 06043088** | ELEC70126 Internet of Things and Applications

Analysis cutoff: **22 March 2026** (17+ days of data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 150

IMAGE_DIR = '../images/'

## 1. Data Loading and Cleaning

In [ ]:
# Load the cutoff dataset (up to 22 March 2026)
df = pd.read_csv('../data/experiment_data_mar22.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
print(f"Duration: {df['Timestamp'].max() - df['Timestamp'].min()}")
print(f"Sampling interval (median): {df['Timestamp'].diff().median()}")
print()
df.describe()

In [ ]:
# Detect and clean anomalies (simultaneous drops across all channels from physical disturbance)
diff_green = df['Green'].diff().abs()
diff_blue = df['Blue'].diff().abs()
diff_control = df['Control'].diff().abs()

THRESHOLD = 200
anomaly_mask = (diff_green > THRESHOLD) & (diff_blue > THRESHOLD) & (diff_control > THRESHOLD)
anomaly_indices = df.index[anomaly_mask].tolist()

rows_to_remove = set()
for idx in anomaly_indices:
    rows_to_remove.add(idx)
    if idx > 0:
        rows_to_remove.add(idx - 1)

df_clean = df.copy()
for idx in rows_to_remove:
    df_clean.loc[idx, ['Green', 'Blue', 'Control']] = np.nan
df_clean[['Green', 'Blue', 'Control']] = df_clean[['Green', 'Blue', 'Control']].interpolate(method='linear')

print(f"Anomalous rows cleaned: {len(rows_to_remove)} (interpolated)")

## 2. Growth Curves (Cleaned Data)

The photoresistor reading **increases** as the plant grows and obstructs more of the light path between the LED and the sensor. A higher ADC reading indicates more growth obstruction.

**Sensor caps:** Blue and Control chambers max out at **4095** ADC. Green chamber caps at **3804** due to ambient green LED interference.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Absolute readings
ax1 = axes[0]
ax1.plot(df_clean['Timestamp'], df_clean['Green'], color='#2ca02c', label='Green Light', linewidth=1.2)
ax1.plot(df_clean['Timestamp'], df_clean['Blue'], color='#1f77b4', label='Blue Light', linewidth=1.2)
ax1.plot(df_clean['Timestamp'], df_clean['Control'], color='#333333', label='Control (Dark)', linewidth=1.2)
ax1.axhline(y=4095, color='red', linestyle=':', alpha=0.5, label='Sensor Cap (Blue/Control: 4095)')
ax1.axhline(y=3804, color='#2ca02c', linestyle=':', alpha=0.5, label='Sensor Cap (Green: 3804)')
ax1.set_ylabel('Photoresistor Reading (ADC)')
ax1.set_title('Bean Sprout Growth Curves — Photoresistor Readings Over Time')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Relative growth from baseline
baseline_green = df_clean['Green'].iloc[:10].mean()
baseline_blue = df_clean['Blue'].iloc[:10].mean()
baseline_control = df_clean['Control'].iloc[:10].mean()

ax2 = axes[1]
ax2.plot(df_clean['Timestamp'], df_clean['Green'] - baseline_green, color='#2ca02c', label=f'Green (baseline={baseline_green:.0f})', linewidth=1.2)
ax2.plot(df_clean['Timestamp'], df_clean['Blue'] - baseline_blue, color='#1f77b4', label=f'Blue (baseline={baseline_blue:.0f})', linewidth=1.2)
ax2.plot(df_clean['Timestamp'], df_clean['Control'] - baseline_control, color='#333333', label=f'Control (baseline={baseline_control:.0f})', linewidth=1.2)
ax2.set_ylabel('Change from Baseline (ADC)')
ax2.set_xlabel('Time')
ax2.set_title('Relative Growth from Baseline')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig(IMAGE_DIR + 'growth_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Harvestability Analysis

Based on manual observation, a bean sprout is considered **100% harvestable** when the sensor reaches a specific threshold:
- **Blue & Control:** ADC ≥ **4000** (sensor cap = 4095)
- **Green:** ADC ≥ **3700** (sensor cap = 3804)

Below these thresholds, harvestability is expressed as a percentage scaled linearly from the baseline to the threshold.

In [ ]:
# Harvestability thresholds (100% harvestable based on manual observation)
HARVEST_THRESH = {'Green': 3700, 'Blue': 4000, 'Control': 4000}

baselines = {
    'Green': df_clean['Green'].iloc[:10].mean(),
    'Blue': df_clean['Blue'].iloc[:10].mean(),
    'Control': df_clean['Control'].iloc[:10].mean()
}

for ch in ['Green', 'Blue', 'Control']:
    bl = baselines[ch]
    thresh = HARVEST_THRESH[ch]
    # Linear scale from baseline (0%) to threshold (100%), capped at 100%
    df_clean[f'{ch}_Harvestability'] = ((df_clean[ch] - bl) / (thresh - bl) * 100).clip(0, 100)

# Plot harvestability
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_clean['Timestamp'], df_clean['Green_Harvestability'], color='#2ca02c', label='Green', linewidth=1.2)
ax.plot(df_clean['Timestamp'], df_clean['Blue_Harvestability'], color='#1f77b4', label='Blue', linewidth=1.2)
ax.plot(df_clean['Timestamp'], df_clean['Control_Harvestability'], color='#333333', label='Control', linewidth=1.2)
ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='100% Harvestable')
ax.set_ylabel('Harvestability (%)')
ax.set_xlabel('Time')
ax.set_title('Bean Sprout Harvestability Over Time')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 110)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'harvestability.png', dpi=150, bbox_inches='tight')
plt.show()

# Print when each chamber first reached 100%
for ch in ['Green', 'Blue', 'Control']:
    full = df_clean[df_clean[f'{ch}_Harvestability'] >= 100]
    if len(full) > 0:
        print(f"{ch}: first 100% harvestable at {full['Timestamp'].iloc[0]} (ADC={full[ch].iloc[0]:.0f})")
    else:
        print(f"{ch}: never reached 100% harvestability in this period")

## 4. Plant Length Estimation

Since there is no direct length sensor, we estimate plant length using a **constant linear growth rate** derived from the harvestability threshold:

1. When a chamber first reaches its **harvestability threshold** (100%), the plant is estimated to be **10 cm** — the minimum height needed to fully block the light path based on chamber geometry.
2. The **growth rate** is constant from the very start: `rate = 10 cm / N`, where N is the number of data points up to and including the threshold crossing.
3. This same rate continues after the threshold, capped at **30 cm** (observed maximum from manual measurement on 24 March).
4. The result is a purely **linear** growth curve per chamber.

Each chamber has its own independent growth rate based on when it crossed its respective threshold.

In [ ]:
# Plant length estimation — constant linear rate
MAX_LENGTH = 30.0  # cm, observed max on 24 March
THRESHOLD_LENGTH = 10.0  # cm, length when sensor fully blocked

for ch in ['Green', 'Blue', 'Control']:
    thresh = HARVEST_THRESH[ch]
    crossed = df_clean[df_clean[ch] >= thresh]
    
    if len(crossed) > 0:
        first_cross_idx = crossed.index[0]
        # Number of data points from start up to and including threshold crossing
        n_points_to_threshold = first_cross_idx + 1  # +1 because index is 0-based
        # Constant growth rate: 10 cm over n_points_to_threshold
        rate = THRESHOLD_LENGTH / n_points_to_threshold
        
        # Apply linear growth from the start, capped at MAX_LENGTH
        df_clean[f'{ch}_Length'] = [(i + 1) * rate for i in range(len(df_clean))]
        df_clean[f'{ch}_Length'] = df_clean[f'{ch}_Length'].clip(upper=MAX_LENGTH)
        
        print(f"{ch}: threshold crossed at index {first_cross_idx} ({df_clean.loc[first_cross_idx, 'Timestamp']})")
        print(f"  N points to threshold: {n_points_to_threshold}")
        print(f"  Constant growth rate: {rate:.4f} cm/point ({rate*4:.4f} cm/hour)")
        print(f"  Final estimated length: {df_clean[f'{ch}_Length'].iloc[-1]:.1f} cm")
    else:
        # Never crossed — use harvestability as rough proxy (0 to 10 cm)
        df_clean[f'{ch}_Length'] = df_clean[f'{ch}_Harvestability'] / 100.0 * THRESHOLD_LENGTH
        print(f"{ch}: never crossed threshold, max estimated length: {df_clean[f'{ch}_Length'].max():.1f} cm")
    print()

In [ ]:
# Plot plant length
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_clean['Timestamp'], df_clean['Green_Length'], color='#2ca02c', label='Green', linewidth=1.5)
ax.plot(df_clean['Timestamp'], df_clean['Blue_Length'], color='#1f77b4', label='Blue', linewidth=1.5)
ax.plot(df_clean['Timestamp'], df_clean['Control_Length'], color='#333333', label='Control', linewidth=1.5)
ax.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='Threshold length (10 cm)')
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='Max length (30 cm)')
ax.set_ylabel('Estimated Plant Length (cm)')
ax.set_xlabel('Time')
ax.set_title('Estimated Bean Sprout Length Over Time')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-1, 33)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'plant_length.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("=== Plant Length Summary ===")
for ch in ['Green', 'Blue', 'Control']:
    print(f"  {ch}: Final = {df_clean[f'{ch}_Length'].iloc[-1]:.1f} cm, Max = {df_clean[f'{ch}_Length'].max():.1f} cm")

## 5. Environmental Conditions

Temperature and humidity measured by the DHT11 sensor throughout the experiment.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(df_clean['Timestamp'], df_clean['Temp(C)'], color='#d62728', linewidth=1)
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Environmental Conditions During Experiment')
ax1.grid(True, alpha=0.3)
ax1.fill_between(df_clean['Timestamp'], df_clean['Temp(C)'].min(), df_clean['Temp(C)'], alpha=0.15, color='red')

ax2.plot(df_clean['Timestamp'], df_clean['Humidity(%)'], color='#17becf', linewidth=1)
ax2.set_ylabel('Humidity (%)')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)
ax2.fill_between(df_clean['Timestamp'], df_clean['Humidity(%)'].min(), df_clean['Humidity(%)'], alpha=0.15, color='cyan')

for ax in [ax1, ax2]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig(IMAGE_DIR + 'environmental_conditions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Temperature — Mean: {df_clean['Temp(C)'].mean():.1f}°C, Min: {df_clean['Temp(C)'].min():.1f}°C, Max: {df_clean['Temp(C)'].max():.1f}°C")
print(f"Humidity    — Mean: {df_clean['Humidity(%)'].mean():.1f}%, Min: {df_clean['Humidity(%)'].min():.1f}%, Max: {df_clean['Humidity(%)'].max():.1f}%")

## 6. Correlation Analysis

Since each chamber is independent, we compute **separate correlation matrices** for each chamber. Each matrix includes:
- The chamber's ADC reading (with harvestability capped at threshold to avoid oscillation artefacts)
- The chamber's estimated plant length
- Temperature and Humidity

**Important:** For correlation purposes, the ADC values are **capped at the harvestability threshold** once reached. This is because after the plant fully blocks the sensor, any subsequent ADC drops are caused by plant oscillation/movement, not actual growth reduction — which would create misleading negative correlation with the monotonically increasing plant length.

In [ ]:
# Prepare capped ADC values for correlation (cap at harvest threshold once reached)
df_corr = df_clean[['Timestamp', 'Temp(C)', 'Humidity(%)']].copy()

for ch in ['Green', 'Blue', 'Control']:
    thresh = HARVEST_THRESH[ch]
    capped = df_clean[ch].copy()
    # Once threshold is reached, cap all subsequent values at the threshold
    first_cross = df_clean[df_clean[ch] >= thresh].index
    if len(first_cross) > 0:
        first_idx = first_cross[0]
        capped.iloc[first_idx:] = capped.iloc[first_idx:].clip(lower=thresh)
    df_corr[f'{ch}_ADC_capped'] = capped
    df_corr[f'{ch}_Length'] = df_clean[f'{ch}_Length']

# Create separate correlation matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

chamber_colors = {'Green': '#2ca02c', 'Blue': '#1f77b4', 'Control': '#555555'}
chamber_titles = {'Green': 'Green Light Chamber', 'Blue': 'Blue Light Chamber', 'Control': 'Control (Dark) Chamber'}

for i, ch in enumerate(['Green', 'Blue', 'Control']):
    cols = [f'{ch}_ADC_capped', f'{ch}_Length', 'Temp(C)', 'Humidity(%)']
    labels = [f'{ch} ADC', f'{ch} Length', 'Temp (°C)', 'Humidity (%)']
    
    corr = df_corr[cols].dropna().corr()
    corr.index = labels
    corr.columns = labels
    
    im = axes[i].imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    axes[i].set_xticks(range(len(labels)))
    axes[i].set_yticks(range(len(labels)))
    axes[i].set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    axes[i].set_yticklabels(labels, fontsize=9)
    axes[i].set_title(chamber_titles[ch], fontsize=11, fontweight='bold')
    
    # Add text annotations
    for row in range(len(labels)):
        for col in range(len(labels)):
            val = corr.values[row, col]
            color = 'white' if abs(val) > 0.5 else 'black'
            axes[i].text(col, row, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.8, label='Pearson Correlation')
plt.suptitle('Correlation Matrices by Chamber (ADC Capped at Harvest Threshold)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'correlation_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Statistical Analysis

### Welch's t-test on Growth Rates

Comparing hourly growth rates between groups to determine if the differences are statistically significant.

In [ ]:
# Compute hourly growth rate (4 samples = 1 hour)
window = 4
for ch in ['Green', 'Blue', 'Control']:
    df_clean[f'{ch}_rate'] = df_clean[ch].diff(window) / window

rates = df_clean[['Green_rate', 'Blue_rate', 'Control_rate']].dropna()

# Welch's t-test
tests = [
    ("Green vs Blue", "Green_rate", "Blue_rate"),
    ("Green vs Control", "Green_rate", "Control_rate"),
    ("Blue vs Control", "Blue_rate", "Control_rate"),
]

print("=== Welch's t-test on Hourly Growth Rates ===\n")
results = []
for name, a, b in tests:
    t, p = stats.ttest_ind(rates[a], rates[b], equal_var=False)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    results.append({"Comparison": name, "t-stat": f"{t:.3f}", "p-value": f"{p:.6f}", "Sig": sig})
    print(f"  {name}: t={t:.3f}, p={p:.6f} {sig}")

print(f"\nMean growth rates (ADC/15min):")
for ch in ['Green', 'Blue', 'Control']:
    r = rates[f'{ch}_rate']
    print(f"  {ch}: {r.mean():.3f} ± {r.std():.3f}")

In [ ]:
# Growth rate distribution box plot
fig, ax = plt.subplots(figsize=(8, 5))
data_to_plot = [rates['Green_rate'], rates['Blue_rate'], rates['Control_rate']]
bp = ax.boxplot(data_to_plot, labels=['Green', 'Blue', 'Control'], patch_artist=True)
colors = ['#2ca02c', '#1f77b4', '#555555']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Growth Rate (ADC/15 min)')
ax.set_title('Growth Rate Distribution by Group')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(IMAGE_DIR + 'growth_rate_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Enriched Dataset

Save the analysis-enriched dataset with harvestability and plant length columns for use in the report and dashboard.

In [ ]:
# Select columns to save
export_cols = ['Timestamp', 'Green', 'Blue', 'Control', 'Temp(C)', 'Humidity(%)',
               'Green_Harvestability', 'Blue_Harvestability', 'Control_Harvestability',
               'Green_Length', 'Blue_Length', 'Control_Length']

df_export = df_clean[export_cols].copy()
df_export.to_csv('../data/experiment_data_enriched.csv', index=False)
print(f"Saved enriched dataset: {df_export.shape[0]} rows, {df_export.shape[1]} columns")
print(f"Columns: {export_cols}")

## 9. Key Findings Summary

1. **Blue light accelerates growth the most**: The blue-light group reached 100% harvestability within ~3 days (ADC ≥ 4000 by 8 March), the fastest among all chambers.

2. **Control (dark) group grows second fastest**: Despite having no directed light, the dark control reached harvestability threshold by ~10 March, consistent with the biological expectation that mung beans grow vigorously in dark conditions.

3. **Green light has the slowest measurable growth**: The green chamber only briefly touched its threshold (3700) once during the period, suggesting green light may slow growth — or that the ambient green LED interference underrepresents actual sensor readings.

4. **Estimated plant lengths**: By 22 March, Blue and Control chambers show estimated lengths approaching 30 cm, while the Green chamber lags significantly behind.

5. **Environmental conditions were stable**: Temperature ~24.6–27.1°C, humidity ~53–62%. Weak correlation with growth confirms light treatment as the primary variable.

6. **Separate correlation matrices** reveal that within each chamber, ADC readings and plant length are strongly correlated (by construction), while environmental factors show weak correlation — confirming light, not temperature/humidity, drives the observed differences.

7. **ADC capping for correlation**: Capping ADC values at the harvest threshold after first crossing eliminates misleading negative correlations caused by post-saturation plant oscillation/movement.

In [ ]:
print("Analysis complete. All figures saved to ../images/")